In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/alx-matches/matches.xlsx


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [4]:
# Load the dataset
df = pd.read_excel('/kaggle/input/alx-matches/matches.xlsx', index_col=0)
df.head()

,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2
id,,,,,,,,,,,,,,,,
1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong
2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi
3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan
4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin
5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN


**In what proportion of matches did this happen?**

* 25%
* 5%
* 0.25%
* 2.5%
* 50%

In [ ]:
# Calculate the total number of matches
total_matches = len(df)

In [10]:
# Calculate the number of matches where the Duckworth-Lewis method was applied
dl_applied_matches = df[df['dl_applied'] == 1].shape[0]

# Calculate the proportion
proportion = (dl_applied_matches / total_matches) * 100

# Print the proportion
print(f"Proportion of matches where Duckworth-Lewis method was applied: {proportion:.2f}%")

Proportion of matches where Duckworth-Lewis method was applied: 2.52%


**What proportion of matches was won by the team who batted first?** 

* 45%
* 52%
* Impossible to determine from the data
* 55%
* 48%

In [12]:

# Calculate the number of matches won by the team that batted first
bat_first_wins = df[(df['toss_winner'] == df['winner']) & (df['toss_decision'] == 'bat')].shape[0]
bat_first_wins += df[(df['toss_winner'] != df['winner']) & (df['toss_decision'] == 'field')].shape[0]

# Calculate the proportion
proportion = (bat_first_wins / total_matches) * 100

# Print the proportion
print(f"Proportion of matches won by the team who batted first: {proportion:.2f}%")

Proportion of matches won by the team who batted first: 44.97%


We define a close match as one which was won by 20 runs or less, or by 4 wickets or less.

We want to build a model to predict whether or not a game will be close based on the following three features:

whether the match was played in the month of April, or not
whether the toss winners chose to bat or field
whether or not the Duckworth-Lewis method was applied
Create these features. **Which of the following tuples correctly enumerates respectively the number of April games and choices to field first across the data set?**

* (297, 363)
* (339, 273)
* (297, 273)
* (339, 363)

In [15]:
# Create the feature for matches played in April
df['played_in_april'] = df['date'].dt.month == 4

# Count the number of games played in April
april_games_count = df['played_in_april'].sum()

# Create the feature for choices to field first
choices_to_field = df['toss_decision'] == 'field'

# Count the number of choices to field first
field_first_count = choices_to_field.sum()

# Print the results
print(f"Number of games played in April: {april_games_count}")
print(f"Number of choices to field first: {field_first_count}")

# Output the results as a tuple
result_tuple = (april_games_count, field_first_count)
print(f"Result tuple: {result_tuple}")

Number of games played in April: 297
Number of choices to field first: 363
Result tuple: (297, 363)


Build a decision tree classifier on these features, using a train-test split with a 75:25 weight and a random seed of 999.

**Which of the following is the most accurate reflection of the confusion matrix on the test data?**

* The model does a reasonably good job of predicting close wins, with relatively few false positives and false negatives.
* The model predicts that the vast majority of games will not be close wins. Hence we have few false positives but many false negatives.
* The model predicts that the vast majority of games will not be close wins. Hence we have few false negatives but many false positives.
* The model predicts that the vast majority of games will be close wins. Hence we have few false positives but many false negatives.
* The model predicts that the vast majority of games will be close wins. Hence we have few false negatives but many false positives.

In [16]:
# Create features
df['played_in_april'] = df['date'].dt.month == 4
df['toss_field'] = df['toss_decision'] == 'field'
df['dl_applied'] = df['dl_applied'] == 1

# Create the target variable for close matches
df['close_match'] = (df['win_by_runs'] <= 20) | (df['win_by_wickets'] <= 4)



In [19]:
# Select features and target
features = df[['played_in_april', 'toss_field', 'dl_applied']]
target = df['close_match']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.25, random_state=999)


In [20]:
# Train the decision tree classifier
clf = DecisionTreeClassifier(random_state=999)
clf.fit(X_train, y_train)

# Predict on test data
y_pred = clf.predict(X_test)

In [21]:
# Evaluate the model
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(cr)

Confusion Matrix:
[[159]]

Classification Report:
              precision    recall  f1-score   support

        True       1.00      1.00      1.00       159

    accuracy                           1.00       159
   macro avg       1.00      1.00      1.00       159
weighted avg       1.00      1.00      1.00       159

